# Thompson Sampling Helper

Convert engagement into a binary success signal and sample the next variant.


In [ ]:
import numpy as np
import pandas as pd

RESULTS_PATH = "data/experiment_results.csv"
METRIC_COLUMNS = ["likes", "replies", "reposts", "bookmarks"]

df = pd.read_csv(RESULTS_PATH)
df["engagement"] = df[METRIC_COLUMNS].sum(axis=1)
df.head()


The success threshold is the observed median engagement. This is a simple
bandit-friendly conversion, not the primary statistical test.


In [ ]:
threshold = df["engagement"].median()
df["success"] = (df["engagement"] >= threshold).astype(int)

print("Success threshold:", threshold)
df[["tweet_number", "variant", "engagement", "success"]].head()


In [ ]:
counts = (
    df.groupby("variant")["success"]
    .agg(successes="sum", trials="count")
    .astype(int)
)
counts["failures"] = counts["trials"] - counts["successes"]
counts


In [ ]:
def thompson_sampling(counts, seed=42):
    rng = np.random.default_rng(seed)
    samples = {}

    for variant, row in counts.iterrows():
        alpha = row["successes"] + 1
        beta = row["failures"] + 1
        samples[variant] = rng.beta(alpha, beta)

    best_variant = max(samples, key=samples.get)
    return best_variant, samples

variant, samples = thompson_sampling(counts)

print("Sampled probabilities:", samples)
print("Next tweet variant:", variant)
